# AI assist proposal demo

This notebook uses a fake Responses API client so it can run without an API key. The AI assist waits for stable input, creates a proposal, and does not apply it until you accept it.

In [1]:
import json
from types import SimpleNamespace

from aipywidgets import AIAssistTool, AIConfig, AIForm, Action, WhenIdle, fields
from aipywidgets.ai_tools import ToolApprovalRequest, default_ai_tools

In [2]:
fake_identifier_tool = AIAssistTool(
    name="prepareResolveIdentifierMetadata",
    description="Prepare a DOI metadata lookup proposal.",
    parameters={
        "type": "object",
        "properties": {"identifier": {"type": "string"}},
        "required": ["identifier"],
        "additionalProperties": False,
    },
    proposal_builder=lambda args: ToolApprovalRequest(
        message=f"Fetch metadata for {args['identifier']} after approval.",
        preview={"tool": "prepareResolveIdentifierMetadata", "identifier": args["identifier"]},
    ),
    executor=lambda args: {
        "status": "completed",
        "type": "resolved_identifier_metadata",
        "identifier": args["identifier"],
        "title": "FAIR metadata workflows for notebook-based data entry",
    },
)


class FakeResponses:
    def __init__(self):
        self.calls = 0

    def create(self, **kwargs):
        self.calls += 1
        input_items = kwargs.get("input", [])
        latest_user_message = next(
            (
                item.get("content", "")
                for item in reversed(input_items)
                if item.get("role") == "user"
            ),
            "",
        )
        latest_tool_output = next(
            (
                json.loads(item["output"])
                for item in reversed(input_items)
                if item.get("type") == "function_call_output"
            ),
            None,
        )

        if latest_tool_output and latest_tool_output.get("type") == "resolved_identifier_metadata":
            title = latest_tool_output.get("title", "")
            proposal = json.dumps(
                {
                    "message": f"Suggested a title from the DOI metadata. (proposal {self.calls})",
                    "operations": [
                        {"op": "set", "path": "title", "value": title}
                    ],
                }
            )
            return SimpleNamespace(
                output=[
                    {
                        "type": "function_call",
                        "name": "propose_form_update",
                        "call_id": f"call_fake_patch_{self.calls}",
                        "arguments": proposal,
                    }
                ]
            )

        if "Assist id: suggest_title_from_doi" in latest_user_message:
            arguments = json.dumps({"identifier": "10.5281/zenodo.1234567"})
            return SimpleNamespace(
                output=[
                    {
                        "type": "function_call",
                        "name": "prepareResolveIdentifierMetadata",
                        "call_id": f"call_fake_tool_{self.calls}",
                        "arguments": arguments,
                    }
                ]
            )

        output_path = (
            "full_width_keywords"
            if "Assist id: suggest_keywords_full_width" in latest_user_message
            else "keywords"
        )
        variants = [
            ("Suggested keywords from the abstract.", ["notebook", "metadata", "demo"]),
            ("Adjusted keywords for research data.", ["research-data", "metadata", "review"]),
            ("Adjusted keywords for repository deposit.", ["repository", "dataset", "deposit"]),
            ("Adjusted keywords for curation workflow.", ["curation", "metadata", "workflow"]),
            ("Adjusted keywords for reproducibility.", ["reproducibility", "dataset", "documentation"]),
            ("Adjusted keywords for notebook-based entry.", ["notebook", "data-entry", "review"]),
        ]
        message, keywords = variants[(self.calls - 1) % len(variants)]
        message = f"{message} (proposal {self.calls})"
        proposal = json.dumps(
            {
                "message": message,
                "operations": [
                    {"op": "set", "path": output_path, "value": keywords}
                ],
            }
        )
        return SimpleNamespace(
            output=[
                {
                    "type": "function_call",
                    "name": "propose_form_update",
                    "call_id": f"call_fake_patch_{self.calls}",
                    "arguments": proposal,
                }
            ]
        )


class FakeClient:
    responses = FakeResponses()


In [3]:
# To try OpenAI instead of FakeClient, uncomment these imports and the ai= line below.
# from getpass import getpass
# from openai import OpenAI


form = AIForm(
    title="AI assist demo",
    ai=AIConfig(client=FakeClient(), model="fake-model", tools=[fake_identifier_tool]),
    # ai=AIConfig(
    #     client=OpenAI(api_key=getpass("OpenAI API key: ")),
    #     model="gpt-5.4-mini",
    #     tools=default_ai_tools(),
    # ),
    style={"margin_bottom": "380px"},
    steps=[
        {
            "id": "main",
            "label": "Main",
            "fields": [
                fields.Text("doi", label="DOI", full_width=True),
                fields.Text("title", label="Title", full_width=True),
                fields.Textarea("abstract", label="Abstract"),
                fields.Tags("keywords", label="Keywords"),
                fields.Textarea("full_width_abstract", label="Abstract (full width)", full_width=True),
                fields.Tags("full_width_keywords", label="Keywords (full width demo)"),
            ],
        }
    ],
    actions=[Action(id="save", label="Save")],
)


form.ai.assist(
    id="suggest_title_from_doi",
    label="Suggest title from DOI",
    watch=["doi"],
    trigger=WhenIdle(ms=1200, min_chars=8),
    prompt="Suggest a concise dataset title for DOI {{ values.doi }}",
    outputs={"title": "A concise title inferred from the DOI metadata"},
)


form.ai.assist(
    id="suggest_keywords",
    label="Suggest keywords",
    watch=["abstract"],
    trigger=WhenIdle(ms=1200, min_chars=20),
    prompt="Suggest keywords for {{ values.abstract }}",
    outputs={"keywords": "A short list of dataset keywords"},
)

form.ai.assist(
    id="suggest_keywords_full_width",
    label="Suggest keywords for full-width field",
    watch=["full_width_abstract"],
    trigger=WhenIdle(ms=1200, min_chars=20),
    prompt="Suggest keywords for {{ values.full_width_abstract }}",
    outputs={"full_width_keywords": "A short list of dataset keywords"},
)


@form.on_action("save")
def save(ctx):
    ctx.info(f"Saved: {ctx.values}")


form

Edit **DOI** and pause to verify a proposal that fills **Title**. Edit **Abstract** and pause to verify the standard right-side bubble. Edit **Abstract (full width)** and pause to verify the below-field bubble. The matching target fields stay unchanged until you press **Accept**.

In [4]:
import io, logging
from aipywidgets import AIConfig, AIForm, Action, WhenIdle, fields

stream = io.StringIO()
handler = logging.StreamHandler(stream)
logger = logging.getLogger("aipywidgets.ai")
old_handlers = logger.handlers[:]
old_level = logger.level
logger.handlers = [handler]
logger.setLevel(logging.WARNING)

class DummyResponses:
    def create(self, **kwargs):
        raise AssertionError("should not call responses.create here")

class DummyClient:
    responses = DummyResponses()

raw_form = AIForm(
    title="Raw",
    ai=AIConfig(client=DummyClient(), model="dummy"),
    steps=[{"id": "main", "label": "Main", "fields": [fields.Textarea("abstract"), fields.Tags("keywords")]}],
    actions=[Action(id="save", label="Save")],
)
raw_form.ai.assist(
    id="suggest_keywords",
    label="Suggest keywords",
    watch=["abstract"],
    trigger=WhenIdle(ms=1200, min_chars=20),
    prompt="Suggest keywords for {{ values.abstract }}",
    outputs={"keywords": "A short list of dataset keywords"},
)
raw_form.set_value("abstract", "This text is long enough to trigger the min_chars guard.")

handler.flush()
output = stream.getvalue()
logger.handlers = old_handlers
logger.setLevel(old_level)
output

'Cannot schedule AI assist before widget() is rendered: suggest_keywords\n'